# 🧪 W3-D5 概念实验：超参数调优与学习率调度

> 配套阅读：`ima/第3周-Day5-训练全流程串联与超参数调优.md`
>
> 超参数是训练的「旋钮」，最关键的是学习率。

## 实验 1：三种学习率调度策略

固定 vs Warmup+Cosine vs Warmup+Linear。Cosine Decay 是标配。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
total, base, warmup = 200, 2e-5, 20
lr_c = np.full(total, base)
lr_s, lr_l = np.zeros(total), np.zeros(total)
for i in range(total):
    if i < warmup: lr_s[i] = lr_l[i] = base*(i+1)/warmup
    else:
        p = (i-warmup)/(total-warmup)
        lr_s[i] = base*0.5*(1+np.cos(np.pi*p)); lr_l[i] = base*(1-p)
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(lr_c, 'b-', lw=2, label='固定 LR')
ax.plot(lr_s, 'r-', lw=2, label='Warmup + Cosine Decay')
ax.plot(lr_l, 'g-', lw=2, label='Warmup + Linear Decay')
ax.axvline(warmup, color='gray', ls='--', alpha=0.5, label='Warmup结束')
ax.set_xlabel('训练步数'); ax.set_ylabel('学习率')
ax.set_title('学习率调度：Cosine Decay 是大模型微调标配')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 实验 2：学习率太大/太小/正好

LR=0.001 学不动，LR=0.2 爆炸，LR=0.01 稳步下降。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
np.random.seed(5); E = 120
def train(lr):
    loss, out = 3.0, []
    for i in range(E):
        g = 0.02+0.008*np.sin(i/8)
        if lr > 0.15: g += 0.4*np.sin(i*lr)
        if loss < 0.8 and lr > 0.15: g += np.random.normal(0, 0.6)
        loss -= lr*g + np.random.normal(0, 0.01*lr)
        loss = max(0.01, loss); out.append(loss)
    return out
fig, ax = plt.subplots(figsize=(10, 5.5))
for lr,lab,c in [(0.001,'LR=0.001（太小）','gray'),(0.01,'LR=0.01（正好）','green'),
                 (0.05,'LR=0.05（偏大）','orange'),(0.2,'LR=0.2（爆炸）','red')]:
    ax.plot(range(E), train(lr), '-', color=c, lw=2, label=lab)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('学习率：宁小勿大，大一步就翻车')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 实验 3：早停法 (Early Stopping)

训练集降，验证集升 → 过拟合。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
np.random.seed(6); epochs = np.arange(1, 51)
train = 2.5*np.exp(-epochs/10)+0.15+0.002*epochs*0.01
val = 2.3*np.exp(-epochs/10)+0.25+0.0004*np.maximum(0,epochs-15)**1.8+np.random.normal(0,0.015,len(epochs))
val = np.maximum(val, 0.2); best = int(np.argmin(val))+1
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, train, 'b-o', lw=2, ms=3, label='训练集 Loss')
ax.plot(epochs, val, 'r-s', lw=2, ms=3, label='验证集 Loss')
ax.axvline(best, color='orange', ls='--', lw=2, label=f'最佳 epoch={best}')
ax.axvspan(best, 50, alpha=0.08, color='red')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('早停法：验证集回升 → 立刻停止')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 实验 4：超参数搜索策略效率

网格 25 次 vs 贝叶斯 9 次收敛。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
np.random.seed(7)
def obj(x,y): return (x-0.3)**2+(y-0.7)**2+0.05*np.sin(5*x)*np.cos(3*y)
xx,yy = np.meshgrid(np.linspace(0,1,80),np.linspace(0,1,80)); zz = obj(xx,yy)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax in axes: ax.contourf(xx,yy,zz,20,cmap='RdYlGn_r',alpha=0.6)
gx,gy = np.meshgrid(np.linspace(0.1,0.9,5),np.linspace(0.1,0.9,5))
axes[0].scatter(gx,gy,c='blue',s=25,zorder=5); axes[0].set_title('网格搜索（25次）')
axes[1].scatter(np.random.rand(25),np.random.rand(25),c='blue',s=25,zorder=5)
axes[1].set_title('随机搜索（25次）')
ox=[0.5,0.4,0.35,0.32,0.28,0.31,0.30,0.30,0.30]
oy=[0.5,0.6,0.65,0.72,0.69,0.71,0.70,0.70,0.70]
axes[2].plot(ox,oy,'k-',alpha=0.3); axes[2].scatter(ox,oy,c=['blue']*3+['orange']*3+['red']*3,s=30,zorder=5)
axes[2].set_title('贝叶斯（9次收敛）')
for ax in axes: ax.set_xlabel('参数1'); ax.set_ylabel('参数2')
plt.suptitle('超参数搜索：贝叶斯最省', fontsize=13); plt.tight_layout(); plt.show()

## 结论

| 问题 | 实验证据 |
|---|---|
| LR 调度 | 实验1：Cosine Decay 标配 |
| LR 大小 | 实验2：宁可小一点 |
| 早停 | 实验3：验证集回升即停 |
| 搜索策略 | 实验4：贝叶斯 > 随机 > 网格 |

→ 配套阅读：`ima/第3周-Day5-训练全流程串联与超参数调优.md`